# Hands-on 34 (Bab 13): Checklist Operasional Sistem IoT-ML

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 13.

Checklist operasional dalam Markdown agar tim bisa menilai kesiapan sistem secara
berulang: sebelum demo, sebelum deployment mini, setelah retraining, atau setelah
perubahan arsitektur.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Generate Checklist

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/generate_operational_checklist.py"], capture_output=True, text=True)
print(r.stdout)

[SUKSES] Checklist disimpan ke: outputs/reports/iot_ml_operational_checklist.md



## 2. Pratinjau Isi Checklist

In [3]:
content = Path("outputs/reports/iot_ml_operational_checklist.md").read_text(encoding="utf-8")
sections = [line for line in content.splitlines() if line.startswith("## ")]
print("\n".join(sections))
print(f"\nTotal butir checklist: {content.count('- [ ]')}")

## A. Data & Sensor
## B. MQTT & Jaringan
## C. Preprocessing
## D. Model
## E. Inferensi
## F. Storage
## G. Dashboard & Alert
## H. Monitoring Drift & Retraining
## I. Stress Test
## Kesimpulan

Total butir checklist: 44


## Penutup

Checklist mencakup 9 kategori (Data & Sensor, MQTT & Jaringan, Preprocessing, Model,
Inferensi, Storage, Dashboard & Alert, Monitoring Drift & Retraining, Stress Test) plus
kesimpulan kesiapan. File tersimpan di `outputs/reports/iot_ml_operational_checklist.md`
dan siap dicetak/dicentang berulang oleh tim operasional.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 36 dalam urutan buku) agar dapat dijadikan panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 7. Insight penting Hands-on 34

Latihan ini mengajarkan bahwa:
- sistem IoT-ML yang baik harus bisa diperiksa secara sistematis,
- operasi yang stabil membutuhkan disiplin, bukan hanya coding,
- dan checklist sederhana sering menjadi pembeda antara proyek mahasiswa
yang rapi dan proyek yang cepat rapuh.

Penutup
Hands-on 32–34 adalah bagian yang membuat bab operasional ini benar-benar
hidup. Pada Hands-on 32, pembaca belajar memonitor perubahan distribusi data
sebagai sinyal awal drift. Pada Hands-on 33, pembaca belajar memakai feature
importance untuk menjelaskan prediksi secara ringan, sesuai ruh explain ability
untuk model pohon/random forest. Pada Hands-on 34, pembaca merangkum
semuanya dalam checklist operasional sistem IoT-ML, sehingga sistem tidak
hanya akurat, tetapi juga lebih siap dipakai, diawasi, dan dipelihara.

Bagian V — Studi Kasus Praktis (Bab 14–15)
Bab 14. Studi Kasus 1–3: Dari Dataset ke Solusi
14.1 Mengapa studi kasus penting di bagian akhir buku?
Setelah pembaca mempelajari fondasi IoT, data sensor, feature engineering,
supervised learning, anomaly detection, forecasting, edge inference, pipeline
end-to-end, serta keamanan dan operasional, ada satu kebutuhan yang sangat
penting: melihat semua potongan itu bekerja bersama dalam konteks masalah
nyata. Itulah peran studi kasus.
Studi kasus mengubah pembelajaran dari:
- “saya tahu cara memakai pandas”
- “saya tahu cara melatih model”
- “saya tahu cara subscribe MQTT”
menjadi:
- “saya bisa membangun solusi kecil yang masuk akal untuk masalah nyata.”
Bab ini ditetapkan sebagai “Dari Dataset ke Solusi”, bukan sekadar kumpulan
eksperimen terpisah. Itu berarti fokusnya bukan lagi pada algoritma sebagai tujuan,
melainkan pada alur pemecahan masalah yang utuh.

14.2 Pola yang dipakai untuk setiap studi kasus
Agar konsisten dan mudah diikuti mahasiswa maupun praktisi, tiap studi kasus di
bab ini memakai struktur yang sama:

masalah
arsitektur
dataset
synthetic data fallback
preprocessing
model
evaluasi
- deployment mini

Mengapa pola ini efektif?
Karena pembaca tidak hanya belajar “model apa yang dipakai”, tetapi juga:
- dari mana data datang,
- apa fallback jika data belum ada,
- bagaimana data dibersihkan,

- bagaimana solusi diuji,
- dan bagaimana solusi dijalankan secara mini di lapangan.
Dengan pola ini, pembaca bisa meniru struktur yang sama untuk kasus lain di luar
buku.

Studi Kasus A — Smart Room Monitoring
14.A.1 Masalah
Banyak ruang kerja, ruang kelas, laboratorium, dan ruang server mengalami
masalah yang tampak sederhana tetapi sebenarnya penting:

ruangan terlalu panas,
kelembaban tidak stabil,
CO₂ meningkat saat ruangan penuh,
pendinginan bekerja lebih keras,
dan pengguna merasa tidak nyaman meskipun sistem tampak “normal”.

Referensi Perros secara eksplisit menempatkan smart home dan
occupancy-aware control sebagai domain IoT penting, termasuk penggunaan CO₂
sensors untuk mendeteksi keberadaan orang dan mengelola kondisi lingkungan.
Referensi lain juga menempatkan smart building sebagai area penting untuk
environment health and safety, security, operating efficiency, dan pengelolaan
energi.
Rumusan masalah
Bagaimana membangun sistem pemantauan ruangan yang:
- membaca suhu, kelembaban, dan CO₂,
- mengklasifikasikan kondisi ruangan menjadi nyaman atau tidak nyaman,
- dan menampilkan hasilnya secara real-time?

14.A.2 Arsitektur
Arsitektur minimal yang realistis:
sensor / simulator → MQTT → Python subscriber → preprocessing →
model klasifikasi → storage → Grafana dashboard
Komponen:

sensor nyata atau dummy data
Mosquitto MQTT
Python untuk parsing dan inferensi
SQLite / InfluxDB
Grafana

Arsitektur ini sangat konsisten dengan arsitektur open source minimal yang Anda
tetapkan sejak awal buku: ESP32/simulator, Mosquitto, Python, InfluxDB/SQLite, dan
Grafana.

14.A.3 Dataset
Opsi 1 — Data sensor nyata
Sensor yang dipakai:
- suhu
- kelembaban
- CO₂
Data bisa berasal dari:
- ESP32 dengan sensor lingkungan
- Raspberry Pi + sensor
- simulator publisher MQTT
Opsi 2 — Dummy data / synthetic data
Bila sensor nyata belum tersedia, gunakan dataset sintetis ruangan seperti yang
sudah dibangun pada bab-bab sebelumnya:

pola suhu harian
kelembaban yang cenderung berlawanan arah dengan suhu
CO₂ naik saat aktivitas meningkat
noise kecil dan beberapa kondisi warning/fault buatan

14.A.4 Synthetic data fallback
Fallback sintetis sangat penting karena:
- tidak semua pembaca punya sensor CO₂,
- tidak semua lab siap dengan perangkat nyata,
- dan pembelajaran harus bisa dimulai tanpa menunggu hardware lengkap.
Dataset fallback yang baik minimal memiliki:
- timestamp
- device_id
- temperature_c
- humidity_pct
- co2_ppm
- label nyaman / tidak_nyaman

Kapan fallback sintetis dipakai?

saat awal pembelajaran
saat pengujian dashboard
saat menguji MQTT → model → dashboard
saat ingin menyuntikkan kondisi ekstrem yang jarang

14.A.5 Preprocessing
Langkah preprocessing yang disarankan:
Langkah 1 — Validasi payload
Periksa:
- field wajib ada
- numerik valid
- rentang wajar
Langkah 2 — Parsing waktu
- ubah timestamp ke datetime
- urutkan data
Langkah 3 — Buat fitur ringan
Misalnya:
- hour
- is_weekend
- temp_lag_1
- co2_lag_1
- temp_ma_3
- co2_ma_3
Langkah 4 — Buat label
Contoh aturan:
- nyaman jika suhu 24–28°C dan kelembaban 45–65% dan CO₂ < 1000
- selain itu → tidak_nyaman
Langkah 5 — Split data time-aware
Pisahkan:
- train
- validation
- test
Ini sangat penting karena sistem ruangan adalah time series.

14.A.6 Model
Model yang paling realistis untuk studi kasus ini:
Model baseline
- Logistic Regression
Model utama
- Random Forest Classifier
Kenapa?

data tabular
fitur mudah dibangun
output kategorikal
feature importance mudah dijelaskan

Target
- nyaman
- tidak_nyaman
Model ini sesuai dengan semangat buku: ringan, open source, cukup kuat, dan
mudah dijelaskan.

14.A.7 Evaluasi
Metrik yang disarankan:

accuracy
precision
recall
F1
confusion matrix

Fokus interpretasi
- apakah tidak_nyaman sering lolos sebagai nyaman?
- apakah alarm terlalu sensitif?
- fitur apa yang paling dominan?

14.A.8 Dashboard real-time
Dashboard Grafana minimal berisi:
- suhu real-time
- kelembaban real-time

CO₂ real-time
status ruangan
histori status
panel warning/alert

14.A.9 Deployment mini
Deployment mini yang realistis:

simulator atau ESP32 mengirim data ke MQTT
Python subscriber di laptop atau Raspberry Pi membaca data
model memprediksi status
hasil ditulis ke InfluxDB
Grafana menampilkan kondisi ruangan

Nilai praktis
Pembaca bisa menggunakannya untuk:

ruang kelas
ruang kerja
lab komputer
ruang server kecil

Studi Kasus B — Predictive Maintenance Sederhana
14.B.1 Masalah
Dalam banyak sistem mekanik atau elektro-mekanik, kerusakan jarang muncul
secara tiba-tiba tanpa tanda. Biasanya ada gejala awal:

getaran meningkat,
arus motor berubah,
suhu motor naik,
suara makin kasar,
pola operasi berubah.

Referensi Perros menempatkan asset maintenance: predictive maintenance to
maximize reliability sebagai salah satu aplikasi utama IoT pada manufaktur.
Referensi Internet of Things and Data Analytics Handbook juga menempatkan
predictive maintenance dan equipment optimization sebagai fungsi inti di smart
manufacturing dan industrial IoT.

Rumusan masalah
Bagaimana membuat sistem sederhana yang:

membaca data getaran dan/atau arus motor,
mendeteksi kondisi abnormal,
memberi klasifikasi fault dasar,
dan membantu interpretasi untuk maintenance?

14.B.2 Arsitektur
Arsitektur minimal:
sensor getaran / arus → MQTT / CSV → Python preprocessing →
anomaly detection + klasifikasi dasar → storage → dashboard / log
maintenance
Komponen:

data getaran atau arus
Python subscriber atau loader CSV
model anomaly detection
model klasifikasi fault dasar
dashboard/terminal/log

14.B.3 Dataset
Opsi 1 — Data nyata
- sensor getaran dari motor/pompa/kipas
- arus listrik motor
- suhu motor jika tersedia
Opsi 2 — Dataset sintetis
Bila alat belum ada:

sinyal getaran sehat vs fault
spike sesaat
amplitudo meningkat
noise lebih besar
arus yang mulai naik saat fault

Data getaran / arus motor digunakan untuk studi kasus ini.

14.B.4 Synthetic data fallback
Fallback sintetis untuk predictive maintenance sangat penting karena:
- data fault nyata sering sangat sedikit,
- mesin rusak tidak mudah “disengaja” hanya untuk latihan,
- banyak pembaca belum punya sensor industri nyata.
Synthetic data fallback sebaiknya mencakup:

sinyal normal
sinyal warning
sinyal fault
injeksi spike
injeksi drift kecil
label normal / warning / fault

14.B.5 Preprocessing
Langkah preprocessing yang disarankan:
Langkah 1 — Windowing
Bagi sinyal menjadi jendela kecil.
Langkah 2 — Ekstraksi fitur
Fitur sederhana yang sangat berguna:
- mean

std
min
max
RMS
peak-to-peak
kurtosis sederhana
arus rata-rata
arus maksimum

Langkah 3 — Validasi sinyal
Periksa:

missing value
jendela kosong
saturasi nilai
outlier sangat ekstrem

Langkah 4 — Label atau skor
Bila ada label:
- gunakan klasifikasi
Bila tidak ada label cukup:
- mulai dari anomaly detection

14.B.6 Model
Pendekatan yang sangat sehat untuk studi kasus ini adalah dua lapis:
Lapis 1 — anomaly detection
Gunakan:
- threshold RMS sederhana
- Isolation Forest
Lapis 2 — klasifikasi fault dasar
Jika label tersedia, gunakan:
- Logistic Regression
- Random Forest Classifier

Kenapa kombinasi ini bagus?
Karena di lapangan:

- data abnormal biasanya lebih mudah dicari dulu,
- lalu fault dasar bisa diklasifikasi bila data dan label bertambah.
Anomaly detection + klasifikasi fault dasar digunakan dalam studi kasus ini.

14.B.7 Evaluasi
Untuk anomaly detection:
- visualisasi sinyal
- confusion matrix bila ground truth sintetis tersedia
- precision / recall terhadap anomali sintetis
Untuk klasifikasi:
- accuracy
- recall untuk fault
- confusion matrix

Fokus penting
Pada maintenance, recall fault sering lebih penting daripada accuracy
semata.

14.B.8 Interpretasi hasil
Interpretasi sederhana yang baik:

“RMS getaran meningkat stabil selama 2 jam terakhir.”
“Fault dasar dipicu terutama oleh kenaikan RMS dan peak-to-peak.”
“Arus dan getaran naik bersamaan, indikasi beban mekanik abnormal.”
“Status warning lebih sering muncul di jam operasi penuh.”

Explain Ability ringan yang cocok
- feature importance
- grafik fitur per jendela
- tabel status dan waktu kejadian

14.B.9 Deployment mini
Deployment mini yang realistis:

data dari CSV / simulator / sensor sederhana
Python membaca data secara batch atau semi-real-time
menghitung fitur jendela
memberi skor anomali
menulis hasil ke log/InfluxDB
dashboard menampilkan histori warning dan fault

Nilai praktis
Pembaca bisa mencoba ini pada:

kipas
motor kecil
pompa air
blower
alat lab

Studi Kasus C — Smart Energy Analytics
14.C.1 Masalah
Salah satu domain IoT yang paling jelas manfaatnya adalah energi. Referensi
menempatkan smart grid, energy management, dan real-time energy
optimization sebagai area utama IoT. Dalam konteks bangunan, laboratorium, atau
fasilitas kecil, pertanyaan utamanya sering berupa:

kapan beban listrik meningkat?
berapa prediksi beban berikutnya?
apakah ada konsumsi tidak normal?
apakah ada pemborosan di luar jam kerja?

Rumusan masalah
Bagaimana membangun sistem yang:
- membaca data konsumsi energi,
- memprediksi beban,
- dan memberi alert saat pola beban tampak tidak normal?

14.C.2 Arsitektur
Arsitektur minimal:
meter energi / simulator → MQTT / CSV → Python preprocessing →
forecasting + anomaly/threshold alert → storage → dashboard
Komponen:

data daya / arus / tegangan
preprocessing waktu
model forecasting
rule/alert beban tidak normal
dashboard energi

14.C.3 Dataset
Opsi 1 — Data nyata
- smart meter
- meter daya lokal
- logger energi
Opsi 2 — Dataset sintetis
Bila belum ada perangkat:

data energi 30 hari
pola jam kerja
jam sibuk
akhir pekan lebih rendah
beberapa lonjakan abnormal

14.C.4 Synthetic data fallback
Fallback sintetis sangat membantu karena:
- smart meter nyata belum selalu tersedia,
- pola jam kerja bisa disimulasikan dengan cukup realistis,
- lonjakan abnormal bisa disuntik untuk menguji alert.
Dataset fallback sebaiknya berisi:
- timestamp
- power_watt
- current_amp opsional
- voltage opsional
- label atau penanda anomali sintetis jika ingin evaluasi mudah

14.C.5 Preprocessing
Langkah preprocessing yang disarankan:
Langkah 1 — Parsing waktu
- urutkan timestamp
- cek interval
Langkah 2 — Buat fitur waktu
- hour
- day_of_week
- is_weekend

- hour_sin, hour_cos
Langkah 3 — Buat fitur historis
- power_lag_1
- power_lag_6
- power_ma_3
- power_ma_6
- power_std_6
- power_diff_1
Langkah 4 — Tentukan target
Untuk forecasting:
- power_watt_next_1step
Untuk alert:
- threshold atau skor anomali

14.C.6 Model
Pendekatan utama untuk studi kasus ini:
Forecasting beban

baseline: last value
baseline: moving average
Linear Regression berbasis lag
Random Forest Regressor

Alert beban tidak normal
- threshold statis
- Isolation Forest opsional
- rule berbasis forecast error bila ingin lanjut
Kenapa pendekatan ini bagus?
Karena energi sangat cocok untuk kombinasi:
- prediksi jangka pendek
- rule operasional yang transparan

14.C.7 Evaluasi
Untuk forecasting:
- MAE
- RMSE
- plot aktual vs prediksi
Untuk alert:
- precision / recall bila ada anomali sintetis
- atau evaluasi operasional berbasis visual dan rule

Hal penting yang perlu dilihat

apakah model menangkap jam sibuk?
apakah akhir pekan terlihat berbeda?
apakah alert terlalu sering palsu?
apakah model masih stabil saat distribusi beban bergeser?

14.C.8 Alert beban tidak normal
Strategi paling realistis:
- threshold untuk alarm jelas, misalnya power_watt > batas
- forecast-aware alert jika nilai aktual jauh di atas prediksi
- hybrid: threshold + deviasi dari baseline

Contoh narasi alert
- “Beban aktual 35% di atas prediksi 10 menit terakhir.”
- “Lonjakan daya muncul di luar jam operasional.”
- “Pola beban tidak sesuai baseline hari kerja.”

14.C.9 Deployment mini
Deployment mini yang realistis:

simulator energi mengirim data ke MQTT
Python subscriber membentuk fitur
model forecasting memprediksi beban berikutnya
hasil disimpan ke InfluxDB
Grafana menampilkan:
- aktual
- prediksi
- alert

Nilai praktis

Studi kasus ini sangat cocok untuk:
- lab komputer
- ruang server

- gedung kecil
- UMKM kecil
- mini smart building

14.10 Benang merah tiga studi kasus
Ketiga studi kasus ini sengaja dipilih karena mewakili tiga pola pemecahan masalah yang
sangat penting dalam IoT analytics:
Smart Room Monitoring
Fokus pada:
- klasifikasi kondisi
- dashboard real-time
- konteks lingkungan dan kenyamanan
Predictive Maintenance Sederhana
Fokus pada:
- anomaly detection
- fault dasar
- interpretasi untuk maintenance
Smart Energy Analytics
Fokus pada:
- forecasting
- operasional energi
- alert beban tidak normal
Dengan demikian, pembaca melihat bahwa IoT-ML tidak hanya satu gaya solusi.
Kadang masalah diselesaikan dengan:

klasifikasi,
kadang dengan anomaly detection,
kadang dengan forecasting,
tetapi semuanya tetap memakai pola rekayasa sistem yang sama.

14.11 Langkah kerja yang harus dilakukan pembaca
Agar ketiga studi kasus ini benar-benar operasional, pembaca dapat mengikuti pola berikut
untuk masing-masing kasus.

Tahap 1 — Definisikan masalah
Tentukan:
- apa yang ingin diprediksi
- apa yang ingin dideteksi
- siapa pengguna hasilnya

Tahap 2 — Susun arsitektur minimal
Pilih:

sensor nyata atau simulator
MQTT / CSV
Python
storage
dashboard

Tahap 3 — Tentukan dataset
Gunakan:
- data nyata jika ada
- synthetic fallback bila belum ada

Tahap 4 — Lakukan preprocessing
Buat:
- validasi
- parsing waktu
- fitur penting

Tahap 5 — Pilih model sesuai masalah
- klasifikasi untuk room monitoring
- anomaly detection + fault dasar untuk maintenance
- forecasting untuk energi

Tahap 6 — Evaluasi
Gunakan metrik yang sesuai dan visualisasi yang jujur.

Tahap 7 — Buat deployment mini
Pastikan solusi benar-benar bisa dijalankan, bukan hanya ada di notebook.

14.12 Sikap profesional dalam menyusun studi kasus
Ada satu pelajaran penting yang perlu ditekankan: studi kasus yang baik bukan
yang paling besar, tetapi yang paling utuh.
Utuh berarti:

masalahnya jelas,
data dan fallback-nya ada,
preprocessing-nya masuk akal,
modelnya cocok,
evaluasinya jujur,
dan deployment mininya benar-benar bisa dijalankan.

Itulah yang membuat studi kasus berguna untuk pembaca dan bisa diulang kembali.

14.13 Penutup bab
Bab 14 adalah titik ketika seluruh isi buku berubah menjadi solusi yang
benar-benar terasa nyata. Melalui Studi Kasus A — Smart Room Monitoring,
pembaca melihat bagaimana data suhu, kelembaban, dan CO₂ dapat diubah menjadi
klasifikasi kenyamanan dan dashboard real-time. Melalui Studi Kasus B —
Predictive Maintenance Sederhana, pembaca memahami bagaimana data getaran
dan arus motor dapat dipakai untuk anomaly detection, klasifikasi fault dasar, dan
interpretasi maintenance. Melalui Studi Kasus C — Smart Energy Analytics,
pembaca melihat bagaimana data konsumsi energi dapat dipakai untuk forecasting
beban dan alert operasional. Ketiganya disusun dengan pola yang sama—masalah,
arsitektur, dataset, synthetic fallback, preprocessing, model, evaluasi,
deployment mini—sehingga pembaca bukan hanya melihat contoh, tetapi juga
belajar kerangka berpikir yang bisa diulang untuk masalah lain.